In [ ]:
# conda activate genomic_tools

import os
import json
import pandas as pd
from collections import defaultdict


## Load interproscan results

In [2]:
cols = [
    'protein_accession',
    'sequence_md5',
    'sequence_length',
    'analysis',
    'signature_accession',
    'signature_description',
    'start',
    'stop',
    'score',
    'status',
    'date',
    'interpro_accession',
    'interpro_description',
    'go_annotations',
    'pathways'
]

interproscan_results = pd.read_csv(
    "data/interproscan/results/proteins.fa.tsv",
    sep='\t',
    header=None,
    names=cols,
    index_col=False
)

In [ ]:
# Parse the PIRSR data

with open("data/interproscan/interpro/data/pirsr/sr_uru.json") as f:
    pirsr_data = json.load(f)

# subset to proteins patterns applicable to humans
human_relevant = ['Eukaryota', 'Eukaryota; Metazoa', 'Eukaryota; Vertebrata', 'Eukaryota; Chordata', 'Eukaryota; Mammalia', 'Eukaryota; Eutheria']

records = []
for ac, entry in pirsr_data.items():
    for group_id, sites in entry['Groups'].items():
        for site in sites:
            scope = entry.get('Scope', [])
            tr = entry.get('TR', '')
            if any(s in human_relevant for s in scope):
                records.append({
                    'accession': ac,
                    'scope':  ', '.join(scope),
                    'TR': tr.split("; ")[1],
                    'label': site['label'],
                    'condition': site['condition'],
                    'desc': site['desc'],
                    'group': group_id
                })

pirsr_df = pd.DataFrame(records)

pirsr_df = pirsr_df.groupby("accession").agg(
    scope=('scope', lambda x: ' | '.join(x.unique())),
    TR=('TR', lambda x: ' | '.join(x.unique())),
    label=('label', lambda x: ' | '.join(x.unique())),
    condition=('condition', lambda x: ' | '.join(x.unique())),
    desc=('desc', lambda x: ' | '.join(x.unique())),
    group=('group', lambda x: ' | '.join(x.unique())),
).reset_index()

In [4]:
interproscan_results = interproscan_results.merge(pirsr_df, left_on="signature_accession", right_on="accession", how="left")

## Load event info.

In [ ]:
# # load splicing event info.
# import pickle

# # note: this dictionary includes ALL events that were emitted, not just significant ones.
# with open('data/event_info.pkl', 'rb') as f:
#     event_info = pickle.load(f)
    
# # # note: this dictionary includes ALL events that were emitted, not just significant ones.
# # with open('data/event_dicts.pkl', 'rb') as f:
# #     event_dicts = pickle.load(f)

In [ ]:
# if an event is cell type-specific, and it has a exon_diff_boundary orexon_diff_junction transcript, I need to know if:

# (1) the transcript was emitted and 
# (2) it is also cell type-specific.

# this is important because it means that any overlapping domains of interest are not necessarily cell type-specific

## Get significant splicing events

(Get the transcript associated with each significant event)

In [ ]:
# get all significant splicing events

signif_events = []
for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        signif_exons_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        for idx, _ in signif_exons_df.iterrows():
            if idx not in signif_events:
                signif_events.append(idx)

In [202]:
event_protein_map

{'ENSG00000187634_ProteinCoding_1': {'inclusion': 'ENST00000616016',
  'aa_start': 263,
  'aa_end': 280,
  'frame_preserving': True,
  'clean_start': False,
  'clean_end': False,
  'real_skip': None,
  'exon_diff_boundary_siblings': [],
  'exon_diff_junction_siblings': [],
  'synthetic_skip': 'ENSG00000187634_ProteinCoding_1_synthetic_skip'},
 'ENSG00000188290_ProteinCoding_1': {'inclusion': 'ENST00000304952',
  'aa_start': 36,
  'aa_end': 67,
  'frame_preserving': True,
  'clean_start': True,
  'clean_end': True,
  'real_skip': 'ENST00000484667',
  'exon_diff_boundary_siblings': [],
  'exon_diff_junction_siblings': [],
  'synthetic_skip': 'ENSG00000188290_ProteinCoding_1_synthetic_skip'},
 'ENSG00000188157_ProteinCoding_1': {'inclusion': 'ENST00000620552',
  'aa_start': 1613,
  'aa_end': 1616,
  'frame_preserving': True,
  'clean_start': True,
  'clean_end': True,
  'real_skip': 'ENST00000379370',
  'exon_diff_boundary_siblings': [],
  'exon_diff_junction_siblings': [],
  'synthetic_s

In [ ]:
# map events to interproscan results

with open('data/event_protein_map.pkl', 'rb') as f:
    event_protein_map = pickle.load(f)

columns = ['protein_accession', 'sequence_length', 'analysis', 
           'signature_description', 'start', 'stop', 'interpro_description']
analyses_to_exclude = ['NCBIFAM', 'SFLD'] # these tools are for full-length protein classification so it doesn't make sense to consider them as overlapping with a single exon
ipr = interproscan_results.loc[~interproscan_results['analysis'].isin(analyses_to_exclude), columns]

def overlaps_skip(df, aa_start):
    return df[(df['start'] <= aa_start) & (df['stop'] >= aa_start)]

# index ipr by protein_accession once
ipr_grouped = {acc: grp for acc, grp in ipr.groupby('protein_accession')}

event_interproscan_map = defaultdict(dict)
  
for ev, rec in event_protein_map.items():
    
    # inclusion: direct lookup instead of boolean mask
    incl_df = ipr_grouped.get(rec['inclusion'])
    if incl_df is None:
        continue
    overlap_df = incl_df[(incl_df['start'] <= rec['aa_end']) & 
                         (incl_df['stop']  >= rec['aa_start'])]
    if overlap_df.empty:
        continue
    event_interproscan_map[ev]['inclusion'] = overlap_df.assign(
        aa_start = rec['aa_start'],
        aa_end = rec['aa_end'],
        exon_cds_start = rec['exon_cds_start'],
        exon_cds_end = rec['exon_cds_end'],
        frame_preserving = rec['frame_preserving'],
        clean_start = rec['clean_start'],
        clean_end = rec['clean_end'],
    ).reset_index(drop=True)

    # real skip
    if rec.get('real_skip'):
        skip_df = ipr_grouped.get(rec['real_skip'])
        if skip_df is not None:
            s = overlaps_skip(skip_df, rec['aa_start'])
            if not s.empty:
                event_interproscan_map[ev]['real_skip'] = s.reset_index(drop=True)

    # synthetic skip
    if rec.get('synthetic_skip'):
        synth_df = ipr_grouped.get(rec['synthetic_skip'])
        if synth_df is not None:
            s = overlaps_skip(synth_df, rec['aa_start'])
            if not s.empty:
                event_interproscan_map[ev]['synthetic_skip'] = s.reset_index(drop=True)

    # junction siblings
    if rec.get('exon_diff_junction_siblings'):
        frames = [ipr_grouped[t] for t in rec['exon_diff_junction_siblings']
                  if t in ipr_grouped]
        if frames:
            event_interproscan_map[ev]['exon_diff_junction_siblings'] = pd.concat(frames, ignore_index=True)

    # boundary siblings
    if rec.get('exon_diff_boundary_siblings'):
        frames = [ipr_grouped[t] for t in rec['exon_diff_boundary_siblings']
                  if t in ipr_grouped]
        if frames:
            event_interproscan_map[ev]['exon_diff_boundary_siblings'] = pd.concat(frames, ignore_index=True)

## Merge interproscan results with significant splicing event info.

In [217]:
# merge cell type-specific events with InterProScan results

df_list = dict() 

for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        ctype = file.split("_exons.csv")[0]
        print(ctype)
        
        signif_events_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        
        event_interproscan_dict = {ev: event_interproscan_map[ev] for ev in signif_events_df.index if ev in event_interproscan_map}

        result = pd.concat(
            [df.assign(event_id=ev, bucket=bucket)
            for ev, buckets in event_interproscan_dict.items()
            for bucket, df in buckets.items()],
            ignore_index=True
        )
        # move event and bucket to front
        cols = ['event_id', 'bucket'] + [c for c in result.columns if c not in ('event_id', 'bucket')]
        result = result[cols]
 
        # restrict to splicing events that overlap with protein domains identified by InterProScan
        df = result.merge(signif_events_df, left_on="event_id", right_index=True)
 
        # summarize interpro results for significant splicing events
        df_list[ctype] = df.groupby(["event_id", "bucket"]).agg(
            r=('r', lambda x: ' | '.join(map(str, x.unique()))),
            is_specific=('is_specific', lambda x: ' | '.join(map(str, x.unique()))),
            Gene=('Gene', lambda x: ' | '.join(x.unique())),
            transcript_id=('protein_accession', lambda x: ' | '.join(x.unique())),
            chr=('chr', lambda x: ' | '.join(x.unique())),
            exon_start=('exon_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_end=('exon_end', lambda x: ' | '.join(map(str, x.unique()))),
            exon_len=('exon_len', lambda x: ' | '.join(map(str, x.unique()))),
            exon_aa_start=('aa_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_aa_end=('aa_end', lambda x: ' | '.join(map(str, x.unique()))),
            domain_start=('start', lambda x: ' | '.join(map(str, x.unique()))),
            domain_stop=('stop', lambda x: ' | '.join(map(str, x.unique()))),
            protein_sequence_length=('sequence_length', lambda x: ' | '.join(map(str, x.unique()))),
            frame_preserving=('frame_preserving', lambda x: ' | '.join(map(str, x.unique()))),
            n_analyses=('analysis', lambda x: len(x.unique())),
            analyses=('analysis', lambda x: ' | '.join(x.unique())),
            signature_descriptions=('signature_description', lambda x: ' | '.join(x.unique())),
            interpro_descriptions=('interpro_description', lambda x: ' | '.join(x.unique()))
        ).reset_index()
        
    break

Oligo


In [224]:
event_info['ENSG00000167755_ProteinCoding_1']

{'meta': {'chrom': 'chr19',
  'strand': '-',
  'es': 50967169,
  'ee': 50967325,
  'gene': 'ENSG00000167755',
  'us_intron_start': 50963550,
  'ds_intron_end': 50968064},
 'cluster_id': 32890,
 'compatible': {'ENST00000376851': {'aa_start': 13,
   'aa_end': 65,
   'coding_nt_length': 157,
   'overlap_type': 'fully_coding',
   'gtf_frame': 2,
   'clean_start': False,
   'clean_end': False,
   'frame_preserving': False,
   'exon_cds_start': 50967169,
   'exon_cds_end': 50967325,
   'transcript_type': 'protein_coding',
   'transcript_tag': 'alternative_5_UTR,basic,appris_principal_1,CCDS'},
  'ENST00000310157': {'aa_start': 13,
   'aa_end': 65,
   'coding_nt_length': 157,
   'overlap_type': 'fully_coding',
   'gtf_frame': 2,
   'clean_start': False,
   'clean_end': False,
   'frame_preserving': False,
   'exon_cds_start': 50967169,
   'exon_cds_end': 50967325,
   'transcript_type': 'protein_coding',
   'transcript_tag': 'basic,Ensembl_canonical,GENCODE_Primary,MANE_Select,appris_principal

In [222]:
df_list[ctype][df_list[ctype]['bucket'] == "inclusion"].sort_values("n_analyses", ascending=False).head(5)

,event_id,bucket,r,is_specific,Gene,transcript_id,chr,exon_start,exon_end,exon_len,start,stop,aa_start,aa_end,protein_sequence_length,frame_preserving,n_analyses,analyses,signature_descriptions,interpro_descriptions
4369,ENSG00000167755_ProteinCoding_1,inclusion,-0.2365883174223526,False,KLK6,ENST00000310157,chr19,50967169,50967325,157,42 | 23 | 18 | 21 | 19 | 46 | 20 | 17 | 40 | 1...,238 | 126 | 240 | 239 | 242 | 237 | 241 | 202 ...,13.0,65.0,244,False,13,CATH-Gene3D | CATH-FunFam | CDD | PIRSR | Pfam...,Trypsin-like serine proteases | kallikrein-6 i...,"- | Serine proteases, trypsin domain | Peptida..."
5164,ENSG00000196159_ProteinCoding_1,inclusion,0.2141789802334368,False,FAT4,ENST00000394329,chr4,125316400,125321586,5187,1526 | 248 | 578 | 888 | 40 | 1629 | 685 | 354...,1622 | 353 | 684 | 991 | 133 | 1738 | 788 | 47...,0.0,1724.0,4983,True,12,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,Cadherins | Cadherin EGF LAG seven-pass G-type...,- | Cadherin-like | Cadherin-like superfamily ...
1699,ENSG00000111405_ProteinCoding_2,inclusion,0.1932963148508996,False,ENDOU,ENST00000422538,chr12,47720753,47720875,123,16 | 22 | 14 | 1 | 19 | 20 | 21 | 38,67 | 60 | 18 | 410 | 62 | 63 | 58,18.0,59.0,410,True,11,CATH-Gene3D | CATH-FunFam | Pfam | Phobius | S...,- | Poly(U)-specific endoribonuclease | Somato...,- | Somatomedin B domain | Somatomedin B-like ...
5404,ENSG00000204655_ProteinCoding_1,inclusion,-0.2053420856077213,False,MOG,ENST00000376917,chr6,29659319,29659666,348,27 | 32 | 40 | 23 | 30 | 1 | 48 | 38 | 33 | 46,155 | 151 | 144 | 29 | 147 | 129 | 145,29.0,145.0,247,True,11,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,Immunoglobulins | Myelin-oligodendrocyte glyco...,Immunoglobulin-like fold | - | Immunoglobulin ...
3527,ENSG00000149131_ProteinCoding_3,inclusion,0.2115503770876241,False,SERPING1,ENST00000378323,chr11,57599864,57600377,514,157 | 141 | 22 | 25 | 70 | 72 | 151 | 17 | 1 |...,444 | 306 | 501 | 48 | 36 | 128 | 124 | 503 | ...,17.0,188.0,505,False,11,CATH-Gene3D | CATH-FunFam | CDD | MobiDB-lite ...,"Antithrombin, subunit I, domain 2 | Serpin fam...","Serpin superfamily, domain 1 | - | Serpin doma..."


In [198]:
signif_interproscan_results

,event_id,bucket,protein_accession,sequence_length,analysis,signature_description,start,stop,interpro_description,Gene,...,r_diff_Astro,fdr_diff_Astro,r_diff_Micro/PVM,fdr_diff_Micro/PVM,r_diff_VLMC,fdr_diff_VLMC,r_diff_Endo,fdr_diff_Endo,r_diff_Peri,fdr_diff_Peri
0,ENSG00000107331_ProteinCoding_2,inclusion,ENST00000341511,2436,Phobius,Transmembrane region,54,71,-,ABCA2,...,0.472258,0.0,0.717356,0.0,0.422339,0.0,0.272528,0.0,0.398776,0.0
1,ENSG00000107331_ProteinCoding_2,inclusion,ENST00000341511,2436,Phobius,Cytoplasmic domain,43,53,-,ABCA2,...,0.472258,0.0,0.717356,0.0,0.422339,0.0,0.272528,0.0,0.398776,0.0
2,ENSG00000107331_ProteinCoding_2,real_skip,ENST00000371605,2435,Phobius,Cytoplasmic domain,43,53,-,ABCA2,...,0.472258,0.0,0.717356,0.0,0.422339,0.0,0.272528,0.0,0.398776,0.0
3,ENSG00000107331_ProteinCoding_2,synthetic_skip,ENSG00000107331_ProteinCoding_2_synthetic_skip,2435,Phobius,Cytoplasmic domain,43,53,-,ABCA2,...,0.472258,0.0,0.717356,0.0,0.422339,0.0,0.272528,0.0,0.398776,0.0
4,ENSG00000203485_ProteinCoding_12,inclusion,ENST00000392634,1249,MobiDB-lite,Consensus disorder prediction,1112,1249,-,INF2,...,0.393148,0.0,0.500471,0.0,0.308706,0.0,0.246732,0.0,0.303452,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64396,ENSG00000136717_ProteinCoding_5,exon_diff_junction_siblings,ENST00000316724,593,SMART,-,17,269,BAR domain,BIN1,...,-0.321063,0.0,-0.646850,0.0,-0.345754,0.0,-0.297019,0.0,-0.301436,0.0
64397,ENSG00000136717_ProteinCoding_5,exon_diff_junction_siblings,ENST00000316724,593,SUPERFAMILY,BAR/IMD domain-like,31,344,AH/BAR domain superfamily,BIN1,...,-0.321063,0.0,-0.646850,0.0,-0.345754,0.0,-0.297019,0.0,-0.301436,0.0
64398,ENSG00000136717_ProteinCoding_5,exon_diff_junction_siblings,ENST00000316724,593,SUPERFAMILY,SH3-domain,515,592,SH3-like domain superfamily,BIN1,...,-0.321063,0.0,-0.646850,0.0,-0.345754,0.0,-0.297019,0.0,-0.301436,0.0
64399,ENSG00000136717_ProteinCoding_5,exon_diff_junction_siblings,ENST00000316724,593,PROSITE profiles,BAR domain profile,29,276,BAR domain,BIN1,...,-0.321063,0.0,-0.646850,0.0,-0.345754,0.0,-0.297019,0.0,-0.301436,0.0


In [192]:
signif_interproscan_results

NameError: name 'signif_interproscan_results' is not defined

In [90]:
df_list['Oligo'][df_list['Oligo']['transcript_id'] == "ENST00000276410"]

,event_id,r,is_specific,Gene,transcript_id,chr,exon_start,exon_end,exon_len,coding_nt_length,sequence_length,frame_preserving,n_analyses,analyses,signature_descriptions,interpro_descriptions
1664,ENSG00000147434_ProteinCoding_1,-0.2154480437006194,False,CHRNA6,ENST00000276410,chr8,42759069,42759113,45,45,494,True,6,CATH-Gene3D | CATH-FunFam | PIRSR | Pfam | Pho...,Neurotransmitter-gated ion-channel ligand-bind...,Neurotransmitter-gated ion-channel ligand-bind...


In [81]:
df_list['CGE_Class'][df_list['CGE_Class']['is_specific'] == "True"].sort_values('n_analyses', ascending=False).head(10)

,event_id,r,is_specific,Gene,transcript_id,chr,exon_start,exon_end,exon_len,coding_nt_length,sequence_length,frame_preserving,n_analyses,analyses,signature_descriptions,interpro_descriptions
773,ENSG00000091513_ProteinCoding_1,0.1668709033258731,True,TF,ENST00000402696,chr3,133748412,133748584,173,173,698,False,11,CATH-Gene3D | CDD | PIRSR | Pfam | Phobius | S...,Periplasmic binding protein-like II | The N-lo...,- | Transferrin-like domain
2771,ENSG00000147434_ProteinCoding_1,0.4408324265163261,True,CHRNA6,ENST00000276410,chr8,42759069,42759113,45,45,494,True,6,CATH-Gene3D | CATH-FunFam | PIRSR | Pfam | Pho...,Neurotransmitter-gated ion-channel ligand-bind...,Neurotransmitter-gated ion-channel ligand-bind...
3475,ENSG00000165264_ProteinCoding_2,-0.4191931634749585,True,NDUFB6,ENST00000379847,chr9,32570960,32571052,93,93,128,True,4,Pfam | Phobius | TMbed | DeepTMHMM,"NADH:ubiquinone oxidoreductase, NDUFB6/B17 sub...","NADH dehydrogenase 1, beta subcomplex, subunit..."
665,ENSG00000084453_ProteinCoding_4,-0.3502160414190283,True,SLCO1A2,ENST00000445053,chr12,21319372,21319579,208,54,71,True,1,Phobius,Signal peptide N-region | Signal Peptide,-
2205,ENSG00000135905_ProteinCoding_2,-0.68093975082271,True,DOCK10,ENST00000535663,chr2,224784737,224784742,6,6,189,True,1,PROSITE profiles,DOCKER domain profile,DOCKER domain
4497,ENSG00000204580_ProteinCoding_14,-0.2769010832337962,True,DDR1,ENST00000376568,chr6,30895404,30895514,111,111,913,True,1,Phobius,Cytoplasmic domain,-


In [ ]:
df_list['Deep_layer_glutamatergic'][df_list['Deep_layer_glutamatergic']['is_specific'] == "True"].sort_values('n_analyses', ascending=False).head(10)

,event_id,r,is_specific,Gene,transcript_id,chr,exon_start,exon_end,exon_len,coding_nt_length,sequence_length,frame_preserving,n_analyses,analyses,signature_descriptions,interpro_descriptions
228,ENSG00000056291_ProteinCoding_1,0.4471791265064418,True,NPFFR2,ENST00000308744,chr4,72128585,72128919,335,328,420,False,11,CATH-Gene3D | CATH-FunFam | CDD | PIRSR | Pfam...,Rhodopsin 7-helix transmembrane proteins | neu...,"- | G protein-coupled receptor, rhodopsin-like..."
3121,ENSG00000177508_ProteinCoding_1,-0.1988403026764951,True,IRX3,ENST00000329734,chr16,54284497,54285613,1117,1117,501,False,10,CATH-Gene3D | CATH-FunFam | COILS | CDD | Mobi...,Homeodomain-like | Iroquois-class homeobox pro...,- | Homeodomain | KN homeodomain | Iroquois-cl...
2207,ENSG00000146904_ProteinCoding_1,0.3414524560565585,True,EPHA1,ENST00000275815,chr7,143394808,143395014,207,207,976,True,9,CATH-Gene3D | CATH-FunFam | PIRSR | Pfam | Pho...,Transferase(Phosphotransferase) domain 1 | Rec...,"- | Serine-threonine/tyrosine-protein kinase, ..."
1976,ENSG00000139880_ProteinCoding_1,-0.4879409185382066,True,CDH24,ENST00000397359,chr14,23051970,23052083,114,114,819,True,8,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,Cadherins | Protocadherin beta 4 | Cadherin ta...,- | Cadherin-like | Cadherin-like superfamily
87,ENSG00000010704_ProteinCoding_7,-0.1190834969825156,True,HFE,ENST00000353147,chr6,26092685,26092960,276,276,168,True,8,CATH-Gene3D | CATH-FunFam | Pfam | Phobius | S...,Immunoglobulins | Major histocompatibility com...,Immunoglobulin-like fold | - | Immunoglobulin ...
1198,ENSG00000115593_ProteinCoding_1,0.2891061506351097,True,SMYD1,ENST00000419482,chr2,88093517,88093555,39,39,490,True,7,CATH-Gene3D | CATH-FunFam | CDD | Pfam | SMART...,SET domain | Histone-lysine N-methyltransferas...,"SET domain superfamily | - | SMYD1, SET domain..."
2352,ENSG00000151914_ProteinCoding_4,-0.525019250063025,True,DST,ENST00000680361,chr6,56529448,56529774,327,327,7818,True,7,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,- | microtubule-actin cross-linking factor 1 |...,- | Spectrin/alpha-actinin | Spectrin repeat
3105,ENSG00000176884_ProteinCoding_1,0.3713913965532838,True,GRIN1,ENST00000371560,chr9,137148152,137148214,63,63,906,True,6,CATH-FunFam | CDD | PIRSR | Pfam | Phobius | S...,"glutamate receptor ionotropic, NMDA 1 isoform ...","- | Glutamate [NMDA] receptor subunit 1-like, ..."
3627,ENSG00000243955_ProteinCoding_1,0.3461300290715827,True,GSTA1,ENST00000334575,chr6,52796182,52796314,133,133,222,False,6,CATH-Gene3D | CDD | SUPERFAMILY | CATH-FunFam ...,"- | Glutaredoxin | C-terminal, alpha helical d...",- | Thioredoxin-like superfamily | Glutathione...
2197,ENSG00000146122_ProteinCoding_1,0.2245671444135169,True,DAAM2,ENST00000633794,chr6,39886421,39886447,27,27,1077,True,6,CATH-FunFam | Pfam | SMART | SUPERFAMILY | PRO...,Dishevelled associated activator of morphogene...,"- | Formin, FH2 domain | Formin, FH2 domain su..."


In [ ]:
event_protein_map

{'ENSG00000187634_ProteinCoding_1': {'inclusion': 'ENST00000616016',
  'aa_start': 263,
  'aa_end': 280,
  'frame_preserving': True,
  'clean_start': False,
  'clean_end': False,
  'real_skip': 'ENST00000618181',
  'exon_diff_boundary_siblings': [],
  'exon_diff_junction_siblings': [],
  'synthetic_skip': 'ENSG00000187634_ProteinCoding_1_synthetic_skip'},
 'ENSG00000188290_ProteinCoding_1': {'inclusion': 'ENST00000304952',
  'aa_start': 36,
  'aa_end': 67,
  'frame_preserving': True,
  'clean_start': True,
  'clean_end': True,
  'real_skip': 'ENST00000484667',
  'exon_diff_boundary_siblings': [],
  'exon_diff_junction_siblings': [],
  'synthetic_skip': 'ENSG00000188290_ProteinCoding_1_synthetic_skip'},
 'ENSG00000188157_ProteinCoding_1': {'inclusion': 'ENST00000620552',
  'aa_start': 1613,
  'aa_end': 1616,
  'frame_preserving': True,
  'clean_start': True,
  'clean_end': True,
  'real_skip': 'ENST00000379370',
  'exon_diff_boundary_siblings': [],
  'exon_diff_junction_siblings': [],
 

In [ ]:
for ev, rec in event_info.items():
    if "ENST00000530869" in rec['exon_diff_junction']:
        print(rec['compatible'])
        break

{'ENST00000276410': {'aa_start': 73, 'aa_end': 87, 'coding_nt_length': 45, 'overlap_type': 'fully_coding', 'gtf_frame': 0, 'clean_start': True, 'clean_end': True, 'frame_preserving': True, 'exon_cds_start': 42759069, 'exon_cds_end': 42759113, 'transcript_type': 'protein_coding', 'transcript_tag': 'basic,Ensembl_canonical,GENCODE_Primary,MANE_Select,appris_principal_1,CCDS', 'event_id': 'ENSG00000147434_ProteinCoding_1', 'transcript_id': 'ENST00000276410'}, 'ENST00000533810': {'aa_start': 0, 'aa_end': 8, 'coding_nt_length': 27, 'overlap_type': 'partially_coding', 'gtf_frame': 0, 'clean_start': True, 'clean_end': True, 'frame_preserving': True, 'exon_cds_start': 42759069, 'exon_cds_end': 42759095, 'transcript_type': 'protein_coding', 'transcript_tag': 'mRNA_end_NF,cds_end_NF'}}


In [77]:
rec['compatible']

{'ENST00000359512': {'aa_start': 194,
  'aa_end': 239,
  'coding_nt_length': 136,
  'overlap_type': 'fully_coding',
  'gtf_frame': 1,
  'clean_start': False,
  'clean_end': True,
  'frame_preserving': False,
  'exon_cds_start': 156022699,
  'exon_cds_end': 156022834,
  'transcript_type': 'protein_coding',
  'transcript_tag': 'basic,Ensembl_canonical,GENCODE_Primary,appris_principal_1'},
 'ENST00000479401': {'overlap_type': 'noncoding_or_utr',
  'transcript_type': 'protein_coding_CDS_not_defined',
  'transcript_tag': 'non_canonical_TEC'},
 'ENST00000484415': {'overlap_type': 'noncoding_or_utr',
  'transcript_type': 'retained_intron',
  'transcript_tag': ''}}

## Visualize interesting hits

In [ ]:
import pickle
from collections import Counter
from collections import defaultdict
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
"""
plot_clusters.py

Genome-browser-style visualization of one splicing cluster.
Left panel  : exon/intron structure per event.
Right panel : per-cell-type r-value heatmap (from signif_info).

Usage
-----
from plot_clusters import plot_cluster

plot_cluster(cid, members[cid], event_info, signif_info)
plot_cluster(cid, members[cid], event_info, signif_info, save_path='cluster_5178.pdf')

signif_info format
------------------
signif_info[event_id]['gene_name'] = str                                  # store once per event
signif_info[event_id][cell_type]   = {'r': float, 'fdr': float, 'is_specific': bool}

Build it like this:
    for idx, row in signif_exons_df.iterrows():
        signif_info[idx]['gene_name'] = row['Gene']
        signif_info[idx][ct] = {'is_specific': row['is_specific'],
                                 'r': row['r'], 'fdr': row['fdr']}
"""
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
from matplotlib.colors import TwoSlopeNorm


# ── shared row builder (both panels use the same sorted order) ─────────────
def _build_rows(event_ids, event_info, signif_set):
    rows = []
    for ev in event_ids:
        if ev not in event_info:
            continue
        meta = event_info[ev]['meta']
        rows.append({
            'event': ev,
            'chrom': meta['chrom'],
            'gene':  meta.get('gene', ''),
            'es':    meta['es'],
            'ee':    meta['ee'],
            'us':    meta['us_intron_start'],
            'ds':    meta['ds_intron_end'],
            'signif': ev in signif_set,
        })
    rows.sort(key=lambda r: (r['es'], r['ee'], r['ds']))
    return rows


# ── left panel: exon / intron structure ───────────────────────────────────
def _plot_structure(ax, cid, rows, signif_set, signif_info):
    if not rows:
        ax.set_visible(False)
        return

    x_min = min(r['us'] for r in rows)
    x_max = max(r['ds'] for r in rows)
    span  = max(x_max - x_min, 1)
    pad   = span * 0.04

    for i, r in enumerate(rows):
        color = 'tomato' if r['signif'] else 'steelblue'

        # intron span line
        ax.plot([r['us'], r['ds']], [i, i],
                color='#bbbbbb', lw=0.8, zorder=1, solid_capstyle='round')

        # cassette exon block
        width = max(r['ee'] - r['es'], span * 0.003)
        ax.add_patch(Rectangle((r['es'], i - 0.3), width, 0.6,
                                facecolor=color, edgecolor='none', zorder=2))

        # event label
        parts  = r['event'].split('_')
        label  = '_'.join(parts[-2:]) if len(parts) >= 2 else r['event']
        marker = ' ★' if r['signif'] else ''
        ax.text(x_min - pad, i, label + marker,
                ha='right', va='center', fontsize=5.5,
                color='tomato' if r['signif'] else '#555555')

    # gene display name: scan all rows for first event that has gene_name in signif_info
    gene_display = next(
        (signif_info[r['event']].get('gene_name')
        for r in rows
        if r['event'] in signif_info and signif_info[r['event']].get('gene_name')),
        rows[0]['gene']
    )
    n_sig = sum(r['signif'] for r in rows)
    ax.set_title(
        f"cluster {cid}  |  {gene_display}  |  {rows[0]['chrom']}\n"
        f"{len(rows)} events  ({n_sig} significant in ≥1 cell type)",
        fontsize=6.5, loc='left', pad=3
    )
    ax.set_xlim(x_min - pad * 2, x_max + pad)
    ax.set_ylim(-0.8, len(rows) - 0.2)
    ax.set_yticks([])

    def _kb(x, _):
        return f'+{(x - x_min) / 1000:.1f}kb' if x != x_min else '0'
    ax.xaxis.set_major_formatter(plt.FuncFormatter(_kb))
    ax.xaxis.set_major_locator(plt.MaxNLocator(4, integer=False))
    ax.tick_params(axis='x', labelsize=5.5, pad=1)
    for spine in ('top', 'left', 'right'):
        ax.spines[spine].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5)


# ── right panel: per-cell-type r-value heatmap ────────────────────────────
def _plot_celltype_panel(ax, rows, signif_info, cell_types):
    norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
    cmap = plt.get_cmap('RdBu_r')

    for i, r in enumerate(rows):
        ev = r['event']
        for j, ct in enumerate(cell_types):
            info = signif_info.get(ev, {}).get(ct)
            if isinstance(info, dict):
                color = cmap(norm(float(np.clip(info['r'], -1, 1))))
                ec = 'black' if info.get('is_specific') else 'white'
                lw = 1.2    if info.get('is_specific') else 0.3
            else:
                color, ec, lw = '#eeeeee', 'white', 0.3

            ax.add_patch(Rectangle((j, i - 0.4), 1, 0.8,
                                    facecolor=color, edgecolor=ec,
                                    linewidth=lw, zorder=2))

            # dot for FDR < 0.01
            if isinstance(info, dict) and info.get('fdr', 1) < 0.01:
                ax.text(j + 0.5, i, '·', ha='center', va='center',
                        fontsize=9, color='white', zorder=3)

    ax.set_xlim(0, len(cell_types))
    ax.set_ylim(-0.8, len(rows) - 0.2)
    ax.set_xticks([j + 0.5 for j in range(len(cell_types))])
    ax.set_xticklabels(cell_types, rotation=40, ha='left', fontsize=5.5)
    ax.xaxis.set_tick_params(length=0)
    ax.tick_params(axis='x', pad=1)
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    # colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.5, pad=0.02, aspect=12)
    cbar.set_label('r', fontsize=6)
    cbar.ax.tick_params(labelsize=5)
    cbar.set_ticks([-1, -0.5, 0, 0.5, 1])


# ── main entry point ───────────────────────────────────────────────────────
def plot_cluster(cid, event_ids, event_info, signif_info,
                 figsize=None, save_path=None, dpi=150):
    """
    Parameters
    ----------
    cid           : cluster id (for the title)
    event_ids     : members[cid]
    event_info     : dict  event_id -> annotate_event record
    signif_info   : dict  event_id -> {'gene_name': str, cell_type: {'r','fdr','is_specific'}}
    figsize       : (width, height) in inches; None = auto
    save_path     : file path to save (e.g. 'cluster.pdf'); None = plt.show()
    dpi           : resolution for raster output
    """
    signif_set = set(signif_info.keys())
    rows       = _build_rows(event_ids, event_info, signif_set)

    if not rows:
        print(f"cluster {cid}: no events found in event_info")
        return None

    # cell types: all keys except 'gene_name', skip non-dict values defensively
    cell_types = sorted({
        ct
        for r in rows
        for ct, val in signif_info.get(r['event'], {}).items()
        if ct != 'gene_name' and isinstance(val, dict)
    })

    n      = len(rows)
    height = max(1.8, n * 0.38)

    if cell_types:
        n_ct         = len(cell_types)
        default_size = (7 + n_ct * 0.7, height)
        fig, (ax_l, ax_r) = plt.subplots(
            1, 2,
            figsize=figsize or default_size,
            gridspec_kw={'width_ratios': [4, max(1, n_ct)]}
        )
        _plot_celltype_panel(ax_r, rows, signif_info, cell_types)
        ax_r.set_title('cell-type associations\n(border = specific, · = FDR<0.01)',
                        fontsize=6, loc='left', pad=3)
    else:
        fig, ax_l = plt.subplots(figsize=figsize or (7, height))

    _plot_structure(ax_l, cid, rows, signif_set, signif_info)

    ax_l.legend(handles=[
        mpatches.Patch(facecolor='tomato',    label='significant (any ct)'),
        mpatches.Patch(facecolor='steelblue', label='not significant'),
    ], loc='upper right', fontsize=5.5, frameon=False)

    plt.tight_layout(pad=1.5)

    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f"saved -> {save_path}")
    else:
        plt.show()

    return fig

In [ ]:
event_dicts = pickle.load(open('event_dicts.pkl', 'rb'))
event_info = pickle.load(open('event_info.pkl', 'rb'))

In [ ]:
# Get all significant splicing events
signif_info = defaultdict(dict)
for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        signif_exons_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        ct = file.replace("_exons.csv", "")
        for idx, row in signif_exons_df.iterrows():
            signif_info[idx]['gene_name'] = row['Gene']   # store once at event level
            signif_info[idx][ct] = {'is_specific': row['is_specific'], 'r': row['r'], 'fdr': row['fdr']}

In [ ]:
members = defaultdict(list)
for ed in event_dicts:
    members[ed['cluster_id']].append(ed['event'])

In [ ]:
sizes = Counter(ed['cluster_id'] for ed in event_dicts)

specific_cids = [
    cid for cid, evs in members.items()
    if any(
        any(info['is_specific'] for info in signif_info.get(ev, {}).values()
            if isinstance(info, dict))
        for ev in evs
    )
]

multi = {cid: n for cid, n in sizes.items() if (n > 1) and cid in specific_cids}
sorted_cids = [cid for cid, n in sorted(multi.items(), key=lambda kv: -kv[1])]

In [ ]:
with PdfPages('all_clusters.pdf') as pdf:
    for cid in sorted_cids[:50]:
        print(cid)
        fig = plot_cluster(cid, members[cid], event_info, signif_info)
        if fig is not None:
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)